In [0]:
%run ./schemas


In [0]:
%run ./common


In [0]:
# QA setup: separate schemas and tables, never touch production writes.
import json, uuid, time, traceback
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = "github_health"
repo = "apache/spark"
entity = "issues"
run_date = "2026-10-09"
load_type = "INCREMENTAL"
batch_id = "qa_production_path_" + uuid.uuid4().hex[:12]

# Dedicated QA schemas; all target writes go here.
for schema in ("qa_bronze", "qa_silver", "qa_ops"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

for name, schema in [
    (f"{CATALOG}.qa_bronze.issues", BRONZE_SCHEMA['issues']),
    (f"{CATALOG}.qa_silver.issues", SILVER_SCHEMA['issues']),
    (f"{CATALOG}.qa_ops.pipeline_execution_logs", LOG_SCHEMA),
]:
    if not spark.catalog.tableExists(name):
        spark.createDataFrame([], schema).write.format("delta").saveAsTable(name)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.qa_ops.bronze_quarantine (
    quarantine_id STRING, batch_id STRING, repository STRING, entity STRING,
    source_file STRING, raw_payload STRING, error_type STRING,
    error_reason STRING, load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
) USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.qa_ops.silver_quarantine (
    quarantine_id STRING, batch_id STRING, repository STRING, entity STRING,
    record_key STRING, raw_payload STRING, error_type STRING,
    error_reason STRING, load_timestamp TIMESTAMP, quarantined_at TIMESTAMP
) USING DELTA
""")

# Redirect the existing common.log helper to QA (it reads LOG_TABLE dynamically).
LOG_TABLE = f"{CATALOG}.qa_ops.pipeline_execution_logs"

# Override only the output destination of the existing quarantine MERGE helper.
# Keep the exact same deduplication and MERGE semantics as common.quarantine.
def quarantine(spark, df, layer):
    table = f"{CATALOG}.qa_ops.{layer}_quarantine"
    if not df.take(1):
        return 0
    df = df.dropDuplicates(["quarantine_id"])
    n = df.count()
    values = {col: f"s.`{col}`" for col in df.columns}
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), "t.quarantine_id = s.quarantine_id")
        .whenNotMatchedInsert(values=values)
        .execute())
    return n

# Create five TEST records in a separate folder of the existing Volume.
# This is NOT in any production manifest, and contains no real GitHub accounts.
seed = uuid.uuid4().int % 900000000000 + 1000000000000
common_fields = {
    "number": 1,
    "title": "Synthetic Phase 2 QA record",
    "state": "open",
    "created_at": "2026-10-08T10:00:00Z",
    "updated_at": "2026-10-08T11:00:00Z",
    "_repo": repo,
    "_ingested_at": "2026-10-09T00:00:00Z",
    "_batch_id": batch_id,
}
def rec(**overrides):
    return {**common_fields, **overrides}

records = [
    json.dumps(rec(id=seed)),                               # valid → Bronze & Silver
    json.dumps(rec(id=None)),                               # bad Bronze: missing ID
    json.dumps(rec(id="not-a-number")),                     # bad Bronze: type mismatch
    '{"id":',                                              # bad Bronze: malformed JSON
    json.dumps(rec(id=seed+1, created_at="bad-timestamp")), # Bronze valid, Silver rejected
]

qa_dir = Path(ROOT) / "_qa_tests" / batch_id
qa_dir.mkdir(parents=True, exist_ok=True)
qa_file = qa_dir / "synthetic_issues.jsonl"
with qa_file.open("w", encoding="utf-8") as file:
    file.write("\n".join(records) + "\n")
item = {"repository": repo, "entity": entity, "path": str(qa_file)}

print("QA batch:", batch_id)
print("QA source:", qa_file)
print("Synthetic input lines:", len(records))
print("Production tables and watermarks will not be written.")


In [0]:
# EXACT Bronze/Silver transformation functions from the working single-session driver,
# with ONLY the Delta destinations redirected to QA schemas.
# quarantine() is overridden below to redirect its output to QA tables.
# Cell 5: Single-session transformations. Explicit schemas and same MERGE semantics
# as your working notebooks. No notebook.run, no cache, no serverless-blocked config.

BUSINESS_COLS = {
    "commits": ["sha", "node_id", "commit", "author", "committer", "html_url"],
    "issues": ["id", "number", "node_id", "title", "state", "created_at",
               "updated_at", "closed_at", "user", "labels", "html_url"],
    "pulls": ["id", "number", "node_id", "title", "state", "draft",
              "created_at", "updated_at", "closed_at", "merged_at",
              "user", "merged", "html_url"],
}


def merge_delta_changed(df, table_name):
    """Idempotent upsert; unchanged hash leaves load_timestamp and batch_id intact.

    Schema additions must be approved and handled separately; do not enable a
    forbidden serverless session-level autoMerge setting.
    """
    columns = df.columns
    values = {column: f"s.`{column}`" for column in columns}
    updates = {column: expression for column, expression in values.items()
               if column != "record_key"}
    (DeltaTable.forName(spark, table_name).alias("t")
        .merge(df.alias("s"), "t.record_key = s.record_key")
        .whenMatchedUpdate(
            condition="NOT (t.payload_hash <=> s.payload_hash)",
            set=updates
        )
        .whenNotMatchedInsert(values=values)
        .execute())
    metrics = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").first()["operationMetrics"]
    return (int(metrics.get("numTargetRowsInserted", 0)),
            int(metrics.get("numTargetRowsUpdated", 0)))


def source_keys_from_raw(raw, repo, entity):
    """Keys requested in *this* raw batch, including unchanged previously-loaded keys."""
    id_col = F.col("sha") if entity == "commits" else F.col("id").cast("string")
    good = (
        raw.filter(F.col("_corrupt_record").isNull())
           .filter(id_col.isNotNull() & (F.trim(id_col) != ""))
           .filter(F.col("_repo").isNull() | (F.col("_repo") == repo))
    )
    if entity == "issues":
        good = good.filter(F.col("pull_request").isNull())
    return good.select(
        F.concat_ws("|", F.lit(repo), id_col).alias("record_key")
    ).distinct()


def run_bronze_one(item):
    repo, entity, file_path = item["repository"], item["entity"], item["path"]
    started, execution_id = now(), str(uuid.uuid4())
    count = 0
    t0 = time.perf_counter()
    try:
        raw = (spark.read.schema(RAW_SCHEMAS[entity])
               .option("mode", "PERMISSIVE")
               .option("columnNameOfCorruptRecord", "_corrupt_record")
               .json(file_path))
        count = raw.count()
        raw_id = F.col("sha") if entity == "commits" else F.col("id").cast("string")
        updated = F.col("commit.author.date") if entity == "commits" else F.col("updated_at")
        reason = (F.when(F.col("_corrupt_record").isNotNull(), "BAD_JSON_OR_TYPE")
                  .when(raw_id.isNull() | (F.trim(raw_id) == ""), "MISSING_PRIMARY_ID")
                  .when(F.col("_repo").isNotNull() & (F.col("_repo") != repo),
                        "REPOSITORY_MISMATCH"))
        if entity == "issues":
            reason = reason.when(F.col("pull_request").isNotNull(), "PULL_REQUEST_IN_ISSUES")
        staged = (raw.withColumn("_reason", reason)
                  .withColumn("_entity_id", raw_id)
                  .withColumn("_updated_str", updated))

        payload_cols = [F.col(c) for c in RAW_SCHEMAS[entity].fieldNames()
                        if c != "_corrupt_record"]
        failed = staged.filter(F.col("_reason").isNotNull())
        rejected_payload = F.coalesce(
            F.col("_corrupt_record"), F.to_json(F.struct(*payload_cols))
        )
        quarantined_df = failed.select(
            F.sha2(F.concat_ws("|", F.lit(batch_id), F.lit(repo), F.lit(entity),
                               rejected_payload), 256).alias("quarantine_id"),
            F.lit(batch_id).alias("batch_id"),
            F.lit(repo).alias("repository"),
            F.lit(entity).alias("entity"),
            F.lit(file_path).alias("source_file"),
            rejected_payload.alias("raw_payload"),
            F.col("_reason").alias("error_type"),
            F.concat(F.lit("Failed Bronze validation: "),
                     F.col("_reason")).alias("error_reason"),
            F.current_timestamp().alias("load_timestamp"),
            F.current_timestamp().alias("quarantined_at"),
        )
        quarantined_count = quarantine(spark, quarantined_df, "bronze")

        valid = staged.filter(F.col("_reason").isNull())
        relevant_json = F.to_json(F.struct(*[F.col(c) for c in BUSINESS_COLS[entity]]))
        full_payload = F.to_json(F.struct(*payload_cols))
        source = valid.select(
            F.concat_ws("|", F.lit(repo), F.col("_entity_id")).alias("record_key"),
            F.lit(repo).alias("repository"),
            F.lit(batch_id).alias("batch_id"),
            F.lit(file_path).alias("source_file"),
            F.expr("try_cast(_updated_str as timestamp)").alias("source_updated_at"),
            full_payload.alias("raw_payload"),
            F.sha2(relevant_json, 256).alias("payload_hash"),
            F.current_timestamp().alias("load_timestamp"),
            F.expr("try_cast(_ingested_at as timestamp)").alias("ingested_at"),
        )
        ranking = Window.partitionBy("record_key").orderBy(
            F.col("source_updated_at").desc_nulls_last(),
            F.col("payload_hash").desc()
        )
        source = (source.withColumn("_rn", F.row_number().over(ranking))
                  .filter(F.col("_rn") == 1).drop("_rn"))
        valid_count = source.count()
        inserted, updated_count = (merge_delta_changed(source, f"{CATALOG}.qa_bronze.{entity}")
                                   if valid_count else (0, 0))
        result = {"repository": repo, "entity": entity, "rows_processed": count,
                  "valid_keys": valid_count, "inserted": inserted,
                  "updated": updated_count, "upserted": inserted + updated_count,
                  "quarantined": quarantined_count}
        log(spark, execution_id, batch_id, "raw_to_bronze", "BRONZE",
            run_date, load_type, started, "SUCCESS", count,
            upserted=result["upserted"], quarantined=quarantined_count)
        print(f"BRONZE {repo:30} {entity:8} input={count:<6} "
              f"inserted={inserted:<5} updated={updated_count:<5} "
              f"quarantine={quarantined_count} ({time.perf_counter()-t0:.1f}s)", flush=True)
        # Returning raw input key set permits Silver retry even when Bronze MERGE is a no-op.
        return result, source_keys_from_raw(raw, repo, entity)
    except Exception:
        try:
            log(spark, execution_id, batch_id, "raw_to_bronze", "BRONZE",
                run_date, load_type, started, "FAILED", count, traceback.format_exc())
        except Exception as exc:
            print("BRONZE audit failure:", repr(exc), flush=True)
        raise


def run_silver_one(item, current_batch_keys):
    repo, entity = item["repository"], item["entity"]
    started, execution_id = now(), str(uuid.uuid4())
    count = 0
    t0 = time.perf_counter()
    try:
        # Read latest corresponding Bronze records by the RAW batch's business keys.
        # A no-op Bronze retry retains its original batch_id, so filtering ONLY
        # Bronze.batch_id would incorrectly skip it.
        bronze = (spark.table(f"{CATALOG}.qa_bronze.{entity}")
                  .filter(F.col("repository") == repo)
                  .join(current_batch_keys, "record_key", "left_semi"))
        count = bronze.count()
        if count == 0:
            result = {"repository": repo, "entity": entity, "rows_processed": 0,
                      "valid_keys": 0, "inserted": 0, "updated": 0,
                      "upserted": 0, "quarantined": 0}
            log(spark, execution_id, batch_id, "bronze_to_silver", "SILVER",
                run_date, load_type, started, "SUCCESS", 0)
            print(f"SILVER {repo:30} {entity:8} no valid keys ({time.perf_counter()-t0:.1f}s)", flush=True)
            return result

        parsed = bronze.withColumn("api", F.from_json("raw_payload", RAW_SCHEMAS[entity]))
        ts = lambda expr: F.expr(f"try_cast({expr} as timestamp)")
        if entity == "commits":
            silver = parsed.select(
                "record_key", "repository", F.col("api.sha").alias("commit_sha"),
                F.col("api.author.id").alias("author_user_id"),
                ts("api.commit.author.date").alias("commit_timestamp"),
                F.col("api.commit.message").alias("message"),
                "batch_id", "source_updated_at", "payload_hash",
                F.current_timestamp().alias("load_timestamp"))
            reason = (F.when(F.col("commit_sha").isNull(), "NULL_COMMIT_SHA")
                      .when(F.col("commit_timestamp").isNull(), "INVALID_COMMIT_TIMESTAMP"))
        elif entity == "issues":
            silver = parsed.select(
                "record_key", "repository", F.col("api.id").alias("issue_id"),
                F.col("api.number").alias("issue_number"),
                F.col("api.user.id").alias("author_user_id"),
                F.col("api.title").alias("title"), F.col("api.state").alias("state"),
                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.closed_at").alias("closed_at"),
                "batch_id", "source_updated_at", "payload_hash",
                F.current_timestamp().alias("load_timestamp"))
            reason = (F.when(F.col("issue_id").isNull(), "NULL_ISSUE_ID")
                      .when(F.col("created_at").isNull() | F.col("updated_at").isNull(),
                            "INVALID_TIMESTAMP"))
        else:
            silver = parsed.select(
                "record_key", "repository", F.col("api.id").alias("pr_id"),
                F.col("api.number").alias("pr_number"),
                F.col("api.user.id").alias("author_user_id"),
                F.col("api.title").alias("title"), F.col("api.state").alias("state"),
                F.col("api.draft").alias("draft"),
                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.closed_at").alias("closed_at"),
                ts("api.merged_at").alias("merged_at"),
                "batch_id", "source_updated_at", "payload_hash",
                F.current_timestamp().alias("load_timestamp"))
            reason = (F.when(F.col("pr_id").isNull(), "NULL_PR_ID")
                      .when(F.col("created_at").isNull() | F.col("updated_at").isNull(),
                            "INVALID_TIMESTAMP"))

        staged = silver.withColumn("_reason", reason)
        failed = staged.filter(F.col("_reason").isNotNull())
        q = (failed.join(bronze.select("record_key", "raw_payload"), "record_key", "left")
             .select(
                F.sha2(F.concat_ws("|", F.lit(batch_id), F.lit(entity),
                                   "record_key", "_reason"), 256).alias("quarantine_id"),
                F.lit(batch_id).alias("batch_id"), F.col("repository"),
                F.lit(entity).alias("entity"), F.col("record_key"), F.col("raw_payload"),
                F.col("_reason").alias("error_type"),
                F.concat(F.lit("Silver validation failure: "),
                         F.col("_reason")).alias("error_reason"),
                F.current_timestamp().alias("load_timestamp"),
                F.current_timestamp().alias("quarantined_at"),
             ))
        quarantined_count = quarantine(spark, q, "silver")
        valid = staged.filter(F.col("_reason").isNull()).drop("_reason")
        ranking = Window.partitionBy("record_key").orderBy(
            F.col("source_updated_at").desc_nulls_last(), F.col("payload_hash").desc())
        valid = (valid.withColumn("_rn", F.row_number().over(ranking))
                 .filter(F.col("_rn") == 1).drop("_rn"))
        valid_count = valid.count()
        inserted, updated_count = (merge_delta_changed(valid, f"{CATALOG}.qa_silver.{entity}")
                                   if valid_count else (0, 0))
        result = {"repository": repo, "entity": entity, "rows_processed": count,
                  "valid_keys": valid_count, "inserted": inserted,
                  "updated": updated_count, "upserted": inserted + updated_count,
                  "quarantined": quarantined_count}
        log(spark, execution_id, batch_id, "bronze_to_silver", "SILVER",
            run_date, load_type, started, "SUCCESS", count,
            upserted=result["upserted"], quarantined=quarantined_count)
        print(f"SILVER {repo:30} {entity:8} input={count:<6} "
              f"inserted={inserted:<5} updated={updated_count:<5} "
              f"quarantine={quarantined_count} ({time.perf_counter()-t0:.1f}s)", flush=True)
        return result
    except Exception:
        try:
            log(spark, execution_id, batch_id, "bronze_to_silver", "SILVER",
                run_date, load_type, started, "FAILED", count, traceback.format_exc())
        except Exception as exc:
            print("SILVER audit failure:", repr(exc), flush=True)
        raise

In [0]:
# Run the SAME core Bronze/Silver functions against QA Delta destinations.
# Assert expected business behavior and no production table count changes.

prod_bronze_before = spark.table("github_health.bronze.issues").count()
prod_silver_before = spark.table("github_health.silver.issues").count()

bronze_result, keys = run_bronze_one(item)
silver_result = run_silver_one(item, keys)

assert bronze_result["rows_processed"] == 5, bronze_result
assert bronze_result["quarantined"] == 3, bronze_result
assert bronze_result["valid_keys"] == 2, bronze_result
assert silver_result["rows_processed"] == 2, silver_result
assert silver_result["quarantined"] == 1, silver_result
assert silver_result["valid_keys"] == 1, silver_result

bronze_q = spark.table(f"{CATALOG}.qa_ops.bronze_quarantine").filter(F.col("batch_id") == batch_id)
silver_q = spark.table(f"{CATALOG}.qa_ops.silver_quarantine").filter(F.col("batch_id") == batch_id)
qa_bronze = spark.table(f"{CATALOG}.qa_bronze.issues").filter(F.col("batch_id") == batch_id)
qa_silver = spark.table(f"{CATALOG}.qa_silver.issues").filter(F.col("batch_id") == batch_id)

assert bronze_q.count() == 3
assert silver_q.count() == 1
assert qa_bronze.count() == 2
assert qa_silver.count() == 1

bronze_reasons = {r["error_type"] for r in bronze_q.select("error_type").collect()}
silver_reasons = {r["error_type"] for r in silver_q.select("error_type").collect()}
assert "BAD_JSON_OR_TYPE" in bronze_reasons, bronze_reasons
assert "MISSING_PRIMARY_ID" in bronze_reasons, bronze_reasons
assert "INVALID_TIMESTAMP" in silver_reasons, silver_reasons

# Replay to validate idempotent writes and quarantine deduplication.
bronze_again, keys_again = run_bronze_one(item)
silver_again = run_silver_one(item, keys_again)
assert bronze_again["upserted"] == 0, bronze_again
assert silver_again["upserted"] == 0, silver_again
assert bronze_q.count() == 3, "Bronze quarantine duplicated on replay"
assert silver_q.count() == 1, "Silver quarantine duplicated on replay"
assert qa_bronze.count() == 2, "Bronze records duplicated on replay"
assert qa_silver.count() == 1, "Silver records duplicated on replay"

# Verify that no production Bronze/Silver rows changed.
assert spark.table("github_health.bronze.issues").count() == prod_bronze_before
assert spark.table("github_health.silver.issues").count() == prod_silver_before

print("\n================ QA INTEGRATION PASSED ================")
print("Batch:", batch_id)
print("Synthetic JSONL rows: 5")
print("Bronze: 2 accepted, 3 quarantined")
print("Silver: 1 accepted, 1 quarantined")
print("Quarantine IDs remained unique on replay")
print("Unchanged MERGE replays: 0 Bronze and 0 Silver upserts")
print("Production Bronze / Silver row counts: unchanged")
print("Production RAW watermarks: never accessed or modified")
print("=======================================================")
